# 🔍 Module 08: Retrievers & RAG (Retrieval-Augmented Generation)

---

## What is RAG?

**RAG (Retrieval-Augmented Generation)** is the technique of giving LLMs access to external knowledge by:

1. **Retrieving** relevant documents based on the user query
2. **Augmenting** the prompt with those documents
3. **Generating** an answer grounded in the retrieved context

```
User Question
     ↓
Embed Question → Vector Search → Relevant Docs
                                      ↓
              LLM Prompt = "Answer using: [docs] + Question"
                                      ↓
                              Grounded Answer
```

### Why RAG instead of fine-tuning?

| | RAG | Fine-tuning |
|-|-----|-------------|
| **Cost** | Low | Very high |
| **Update data** | Easy (re-index) | Must retrain |
| **Transparency** | Can show sources | Black box |
| **Hallucination** | Reduced | Can increase |
| **Setup time** | Hours | Days/Weeks |

---

In [3]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Setup complete ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Setup complete ✅


## 1️⃣ Simple RAG Pipeline

In [4]:
# ============================================================
# Step 1: Prepare the knowledge base
# ============================================================
# In a real app, you'd load PDFs, web pages, etc.
# Here we use sample documents about a fictional company

company_docs = [
    Document(
        page_content="NexaAI was founded in 2020 by Dr. Sarah Chen and Marcus Thompson in San Francisco. The company focuses on enterprise AI solutions.",
        metadata={"source": "company_overview.txt", "section": "history"}
    ),
    Document(
        page_content="NexaAI's flagship product is NexaChat, an AI-powered customer service platform. It integrates with Salesforce, Zendesk, and Slack. Pricing starts at $499/month.",
        metadata={"source": "products.txt", "section": "products"}
    ),
    Document(
        page_content="NexaAI currently employs 150 people across 3 offices: San Francisco (HQ), New York, and London. We are hiring for ML Engineer and Product Manager roles.",
        metadata={"source": "hr_info.txt", "section": "employment"}
    ),
    Document(
        page_content="NexaAI's refund policy: Customers can request refunds within 30 days of purchase. Enterprise contracts have custom terms. Contact billing@nexaai.com for assistance.",
        metadata={"source": "policies.txt", "section": "billing"}
    ),
    Document(
        page_content="Technical requirements: NexaChat requires Python 3.8+, 4GB RAM minimum, and a stable internet connection. Available as cloud SaaS or on-premise deployment.",
        metadata={"source": "tech_specs.txt", "section": "technical"}
    ),
    Document(
        page_content="NexaAI raised $45M in Series B funding in 2023 led by Andreessen Horowitz. Total funding to date is $62M. Revenue was $8M ARR in 2023.",
        metadata={"source": "investor_info.txt", "section": "finance"}
    )
]

# Create vector store
vectorstore = FAISS.from_documents(company_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Knowledge base ready with {len(company_docs)} documents ✅")

Knowledge base ready with 6 documents ✅


In [5]:
# ============================================================
# Step 2: Build the RAG prompt
# ============================================================
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are a helpful assistant for NexaAI company. 
    Answer questions using ONLY the context provided below.
    If the answer isn't in the context, say "I don't have that information."
    Always be accurate and cite the source when relevant.
    
    Context:
    {context}
    """),
    ("human", "{question}")
])

# Helper to format retrieved docs
def format_docs(docs):
    return "\n\n".join(
        f"[{doc.metadata['source']}]\n{doc.page_content}" 
        for doc in docs
    )

# ============================================================
# Step 3: Assemble the RAG chain
# ============================================================
rag_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever | format_docs,
        question=lambda x: x["question"]
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled ✅")

RAG chain assembled ✅


In [6]:
# ============================================================
# Step 4: Ask questions!
# ============================================================
questions = [
    "Who founded NexaAI and when?",
    "What is the pricing for NexaChat?",
    "How many employees does NexaAI have?",
    "What is the refund policy?",
    "What is the CEO's salary?"  # Not in our docs
]

for question in questions:
    print(f"\n❓ Question: {question}")
    answer = rag_chain.invoke({"question": question})
    print(f"💬 Answer: {answer}")
    print("-" * 60)


❓ Question: Who founded NexaAI and when?
💬 Answer: NexaAI was founded by Dr. Sarah Chen and Marcus Thompson in 2020, according to the company overview (company_overview.txt).
------------------------------------------------------------

❓ Question: What is the pricing for NexaChat?
💬 Answer: The pricing for NexaChat starts at $499/month. (Source: products.txt)
------------------------------------------------------------

❓ Question: How many employees does NexaAI have?
💬 Answer: NexaAI currently employs 150 people. [Source: hr_info.txt]
------------------------------------------------------------

❓ Question: What is the refund policy?
💬 Answer: According to the [policies.txt] document, NexaAI's refund policy is as follows: Customers can request refunds within 30 days of purchase. However, it's noted that Enterprise contracts have custom terms. For assistance, customers can contact billing@nexaai.com.
------------------------------------------------------------

❓ Question: What is th

## 2️⃣ RAG with Source Citations

In [7]:
from pydantic import BaseModel, Field
from typing import List

# ============================================================
# Structured output: answer + sources
# ============================================================
class AnswerWithSources(BaseModel):
    answer: str = Field(description="The answer to the question")
    sources: List[str] = Field(description="List of source files used")
    confidence: str = Field(description="high, medium, or low")

structured_llm = llm.with_structured_output(AnswerWithSources)

sourced_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    Answer the question using the provided context.
    Include the source file names you used in your answer.
    Rate your confidence as high/medium/low.
    If the answer isn't available, say so and rate confidence as low.
    
    Context:
    {context}
    """),
    ("human", "{question}")
])

sourced_chain = (
    RunnableParallel(
        context=(lambda x: x["question"]) | retriever | format_docs,
        question=lambda x: x["question"]
    )
    | sourced_prompt
    | structured_llm
)

result = sourced_chain.invoke({"question": "How much funding has NexaAI raised?"})

print(f"Answer: {result.answer}")
print(f"Sources: {result.sources}")
print(f"Confidence: {result.confidence}")

Answer: $62M
Sources: ['investor_info.txt']
Confidence: high


## 3️⃣ Full RAG Pipeline with Document Loading

In [8]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ============================================================
# Complete RAG pipeline from scratch
# Load → Split → Embed → Store → Retrieve → Generate
# ============================================================

def build_rag_from_urls(urls: list, chunk_size: int = 500, chunk_overlap: int = 50):
    """Build a complete RAG system from web URLs"""
    
    print("📥 Loading documents...")
    loader = WebBaseLoader(urls)
    raw_docs = loader.load()
    print(f"   Loaded {len(raw_docs)} pages")
    
    print("✂️  Splitting into chunks...")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(raw_docs)
    print(f"   Created {len(chunks)} chunks")
    
    print("🔢 Embedding and indexing...")
    vectorstore = FAISS.from_documents(chunks, embeddings)
    print(f"   Indexed {vectorstore.index.ntotal} vectors")
    
    retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
    
    # Build the RAG chain
    prompt = ChatPromptTemplate.from_messages([
        ("system", """Answer the question based on the context below.
        If you don't know, say you don't know.
        
        Context: {context}"""),
        ("human", "{question}")
    ])
    
    chain = (
        RunnableParallel(
            context=(lambda x: x["question"]) | retriever | format_docs,
            question=lambda x: x["question"]
        )
        | prompt | llm | StrOutputParser()
    )
    
    print("✅ RAG system ready!")
    return chain, vectorstore

# Build from LangChain docs
chain, vs = build_rag_from_urls(
    ["https://python.langchain.com/docs/introduction/"]
)

USER_AGENT environment variable not set, consider setting it to identify your requests.


📥 Loading documents...
   Loaded 1 pages
✂️  Splitting into chunks...
   Created 12 chunks
🔢 Embedding and indexing...
   Indexed 12 vectors
✅ RAG system ready!


In [9]:
# Query the RAG system
questions = [
    "What is LangChain used for?",
    "What are the main components of LangChain?",
]

for q in questions:
    print(f"\n❓ {q}")
    print(f"💬 {chain.invoke({'question': q})}")


❓ What is LangChain used for?
💬 LangChain is used for creating agents, which are customizable harnesses that can be tailored to specific use cases and data. It is built on top of LangGraph and allows for the creation of simple or complex agents with various tools and features, such as human-in-the-loop support, persistence, and durable execution. LangChain can be used for a wide range of applications, including natural language processing, automation, and decision-making.

❓ What are the main components of LangChain?
💬 The main components of LangChain are: 

1. Agents
2. Models
3. Messages
4. Tools
5. Short-term memory 
6. Event streaming 
7. Streaming 
8. Structured output
9. Middleware 

These components work together to enable the creation of complex agents and applications with LangChain.


## 4️⃣ Advanced RAG Techniques

In [10]:
# ============================================================
# Technique 1: Hypothetical Document Embeddings (HyDE)
# Generate a hypothetical answer, then use that to retrieve
# ============================================================

# Step 1: Generate a hypothetical answer to improve retrieval
hyde_prompt = ChatPromptTemplate.from_template(
    "Write a short paragraph that would be a perfect answer to this question: {question}"
)

# Step 2: Use that hypothetical answer to retrieve (better semantic match)
hyde_chain = (
    RunnableParallel(
        hypothetical_answer=hyde_prompt | llm | StrOutputParser(),
        original_question=lambda x: x["question"]
    )
    | {
        "context": (lambda x: x["hypothetical_answer"]) | retriever | format_docs,
        "question": lambda x: x["original_question"]
    }
    | rag_prompt | llm | StrOutputParser()
)

# Note: rag_prompt and retriever from earlier cells
# Reinitialize retriever with our company docs
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("HyDE technique ready ✅")

HyDE technique ready ✅


In [14]:
# ============================================================
# Technique 2: Multi-Query Retrieval
# Generate multiple query variations to retrieve more docs
# ============================================================
#from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

# It generates multiple queries from the original!
import logging
logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

unique_docs = multi_query_retriever.invoke("Tell me about NexaAI's products")
print(f"\nMulti-query retrieved {len(unique_docs)} unique documents")


Multi-query retrieved 4 unique documents


In [ ]:
# ============================================================
# Technique 3: Contextual Compression, learn in detail
# Extract only the RELEVANT parts of each retrieved document
# ============================================================
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# Compressor uses LLM to extract relevant parts
compressor = LLMChainExtractor.from_llm(llm)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

query = "What is the employee count?"

# Regular retrieval
regular_docs = retriever.invoke(query)

# Compressed retrieval — extracts only relevant sentences!
compressed_docs = compression_retriever.invoke(query)

print(f"Regular retrieval: {len(regular_docs)} docs")
print(f"\nRegular doc content (first doc, full):\n{regular_docs[0].page_content}")

print(f"\n\nCompressed retrieval: {len(compressed_docs)} docs")
if compressed_docs:
    print(f"Compressed content (extracted):\n{compressed_docs[0].page_content}")

Regular retrieval: 3 docs

Regular doc content (first doc, full):
NexaAI currently employs 150 people across 3 offices: San Francisco (HQ), New York, and London. We are hiring for ML Engineer and Product Manager roles.


Compressed retrieval: 1 docs
Compressed content (extracted):
NexaAI currently employs 150 people across 3 offices: San Francisco (HQ), New York, and London.


## 5️⃣ Conversational RAG — RAG with Memory

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# ============================================================
# Step 1: Rephrase question considering chat history
# ============================================================
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    Given the chat history and latest user question, 
    rephrase the question to be standalone (no pronouns referring to history).
    Do NOT answer the question, just rephrase it if needed.
    """),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# History-aware retriever: rephrases question, then retrieves
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_prompt
)

# ============================================================
# Step 2: Answer with context
# ============================================================
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """
    You are a helpful assistant for NexaAI.
    Answer using the provided context. If unsure, say so.
    
    Context: {context}
    """),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# Combine documents chain
doc_chain = create_stuff_documents_chain(llm, qa_prompt)

# Full retrieval chain
rag_chain = create_retrieval_chain(history_aware_retriever, doc_chain)

# Add message history
session_store = {}

def get_history(session_id):
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

conversational_rag = RunnableWithMessageHistory(
    rag_chain,
    get_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

config = {"configurable": {"session_id": "demo"}}

# Multi-turn conversation with RAG!
conversation = [
    "Who founded NexaAI?",
    "Where did they found it?",     # 'they' refers to founders from previous message
    "How much funding have they raised?",
]

for question in conversation:
    result = conversational_rag.invoke({"input": question}, config=config)
    print(f"\n❓ {question}")
    print(f"💬 {result['answer']}")

## ✅ Module 08 Summary

You've learned:
- ✅ RAG architecture and when to use it (vs fine-tuning)
- ✅ Simple RAG pipeline with LCEL
- ✅ Structured outputs with source citations
- ✅ Full pipeline: Load → Split → Embed → Store → Retrieve → Generate
- ✅ HyDE (Hypothetical Document Embeddings)
- ✅ Multi-query retrieval
- ✅ Contextual compression
- ✅ Conversational RAG with memory

### 🚀 Next: [Module 09 — Tools & Agents](09_Tools_and_Agents.ipynb)